In [22]:
!sudo rm -rf /usr/local/lib/ollama
!sudo rm -f /usr/local/bin/ollama
!rm -f ollama-linux-amd64.tgz ollama

In [26]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [34]:
ollama_model_id = "qwen2.5:7b"
# !pkill -f ollama

In [27]:
import os
# Start Ollama server in the background
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGIN'] = '*'
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
!tail ollama.log

time=2026-06-05T15:00:19.004Z level=INFO source=routes.go:1919 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* ht

In [35]:
# Pull the model first
!ollama pull {ollama_model_id}

# Then test the connection
import requests
import json

url = "http://localhost:11434/api/chat"
payload = {
    "model": ollama_model_id,
    "messages": [{"role": "user", "content": "How many letter r are in strawberry?"}],
    "stream": False
}

try:
    response = requests.post(url, json=payload)
    print(response.json())
except Exception as e:
    print(f"Error: {e}")


{'model': 'qwen2.5:7b', 'created_at': '2026-06-05T15:04:43.910970827Z', 'message': {'role': 'assistant', 'content': 'There are two \'r\' letters in the word "strawberry."'}, 'done': True, 'done_reason': 'stop', 'total_duration': 44761925863, 'load_duration': 21216427878, 'prompt_eval_count': 37, 'prompt_eval_duration': 11310570000, 'eval_count': 16, 'eval_duration': 12229442000}


In [37]:
from google.colab import userdata
ngrok_auth=userdata.get('colab-ngrok')

In [38]:
from pyngrok import ngrok

# Set auth token
ngrok.set_auth_token(ngrok_auth)

# Create tunnel to Ollama port
public_url = ngrok.connect(11434)
print(f"Ollama is now reachable at: {public_url}")

Ollama is now reachable at: NgrokTunnel: "https://8ee9-35-225-28-154.ngrok-free.app" -> "http://localhost:11434"


In [39]:
!pkill -f ollama